<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula12a_ensembles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = fetch_olivetti_faces(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

downloading Olivetti faces from https://ndownloader.figshare.com/files/5976027 to /root/scikit_learn_data


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)
y_pred = rf_classifier.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))

Random Forest Accuracy: 0.925


In [12]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

base_estimator = DecisionTreeClassifier(max_depth=20, splitter="random")
ab_classifier = AdaBoostClassifier(base_estimator)
ab_classifier.fit(X_train, y_train)
y_pred = ab_classifier.predict(X_train)
print("AdaBoost Train Accuracy:", accuracy_score(y_train, y_pred))
y_pred = ab_classifier.predict(X_test)
print("AdaBoost Test Accuracy:", accuracy_score(y_test, y_pred))

AdaBoost Train Accuracy: 1.0
AdaBoost Test Accuracy: 0.9375


In [13]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 10.4 MB/s eta 0:00:00


In [18]:
from sklearn.base import BaseEstimator
from sklearn.model_selection import cross_val_score, StratifiedKFold
import optuna

class OptunaAdaBoost(BaseEstimator):
  def __init__(self , n_trials=100):
    self.n_trials = n_trials
  def fit(self, X, y):
    def objective(trial):
      n_estimators = trial.suggest_int("n_estimators", 10, 100)
      max_depth = trial.suggest_int("max_depth", 1, 10)
      min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
      min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
      base_clf = DecisionTreeClassifier(
          max_depth=max_depth,
          min_samples_split=min_samples_split,
          min_samples_leaf=min_samples_leaf)
      ada_clf = AdaBoostClassifier(estimator=base_clf, n_estimators=n_estimators)
      cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
      scores = cross_val_score(ada_clf, X, y, cv=cv, scoring="accuracy")
      return scores.mean()
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=self.n_trials)
    self.best_params = study.best_params
    base_clf = DecisionTreeClassifier(
        max_depth=self.best_params["max_depth"],
        min_samples_split=self.best_params["min_samples_split"],
        min_samples_leaf=self.best_params["min_samples_leaf"],
        splitter="random"
    )
    self.best_estimator = AdaBoostClassifier(estimator=base_clf,
                                             n_estimators=self.best_params["n_estimators"])
    self.best_estimator.fit(X, y)
  def predict(self, X):
    return self.best_estimator.predict(X)

oab = OptunaAdaBoost(2)
oab.fit(X_train, y_train)
y_pred = oab.predict(X_test)
print("Optuna AdaBoost Accuracy:", accuracy_score(y_test, y_pred))

[I 2026-09-24 15:35:22,086] A new study created in memory with name: no-name-69700c60-5567-437f-b3aa-e3e10951a714
[I 2026-09-24 15:36:23,401] Trial 0 finished with value: 0.4688473520249221 and parameters: {'n_estimators': 22, 'max_depth': 7, 'min_samples_split': 8, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.4688473520249221.
[I 2026-09-24 15:38:15,102] Trial 1 finished with value: 0.40019396931758067 and parameters: {'n_estimators': 58, 'max_depth': 5, 'min_samples_split': 3, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.4688473520249221.


Optuna AdaBoost Accuracy: 0.5875


In [21]:
from sklearn.ensemble import StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron

base_estimators = [
    ("knn", KNeighborsClassifier()),
    ("nb", GaussianNB()),
    ("perceptron", Perceptron())
]
stacking_classifier = StackingClassifier(estimators=base_estimators,
                                         cv=StratifiedKFold(n_splits=3, shuffle=True))
stacking_classifier.fit(X_train, y_train)
y_pred = stacking_classifier.predict(X_train)
print("Stacking Train Accuracy:", accuracy_score(y_train, y_pred))
y_pred = stacking_classifier.predict(X_test)
print("Stacking Accuracy:", accuracy_score(y_test, y_pred))

Stacking Train Accuracy: 0.98125
Stacking Accuracy: 0.8375


In [23]:
from sklearn.linear_model import LogisticRegression

lrc = LogisticRegression()
lrc.fit(X_train, y_train)
y_pred = lrc.predict(X_test)
print("Logistic Regression Train Accuracy:", accuracy_score(y_test, y_pred))

Logistic Regression Train Accuracy: 0.975


In [25]:
from sklearn.model_selection import cross_val_score

lrc = LogisticRegression(tol=0.001)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(lrc, X, y, cv=cv, scoring="accuracy")
print("Logistic Regression Accuracy:", scores.mean())

Logistic Regression Accuracy: 0.9525679871320091


In [28]:
from sklearn.ensemble import VotingClassifier

voting_classifier = VotingClassifier(base_estimators, voting="hard")
scores = cross_val_score(voting_classifier, X, y, cv=cv, scoring="accuracy")
print("Voting Classifier Accuracy:", scores.mean())

Voting Classifier Accuracy: 0.8349605356675269


In [29]:
rfc = RandomForestClassifier()
scores = cross_val_score(rfc, X, y, cv=cv, scoring="accuracy")
print("Random Forest Accuracy:", scores.mean())

Random Forest Accuracy: 0.9075487225526503


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_classifier = GradientBoostingClassifier()
scores = cross_val_score(gb_classifier, X, y, cv=cv, scoring="accuracy")
print("Gradient Boosting Accuracy:", scores.mean())